# 03 · School Choice Through Three Research Personas
**COMSCI/ECON 206 · Computational Microeconomics · Prof. Luyao Zhang**

Use one school-choice problem to connect three questions:

1. **Game theorist:** who can change the outcome by changing only their report?
2. **Social-choice researcher:** which collective goals should the assignment pursue, and where do they conflict?
3. **Mechanism designer:** which rule can implement those goals under strategic behavior?

| Learning path | Do this | Observable evidence |
|---|---|---|
| ① Read the game | Identify players, reports, priorities, capacities, and ordinal payoffs | A minimum viable matching game |
| ② Compare rules | Follow proposals and decisions one round at a time | Boston and deferred-acceptance traces |
| ③ Change one input | Move one student's school to rank 1 in the interactive lab | A before/after allocation and stability check |
| ④ Speak in three personas | Interpret the same result three ways | Strategy, social objective, and implementable rule |

> **Boundary:** a computed example can produce a counterexample; it cannot prove a theorem. Predict first, run both mechanisms, inspect every round, and explain what would overturn your conclusion.

## Intellectual lineage: what each advance made visible

- **Strategic interaction.** Nash formalized mutual best responses; Harsanyi represented incomplete information using types and priors; Selten refined Nash by requiring credible behavior after every relevant history. See the [1994 Prize summary](https://www.nobelprize.org/prizes/economic-sciences/1994/summary/), [Nash (1950)](https://doi.org/10.1073/pnas.36.1.48), [Harsanyi (1967)](https://doi.org/10.1287/mnsc.14.3.159), and [Selten (1965)](https://www.jstor.org/stable/40748884).
- **Social choice.** Arrow's work clarified how individual preferences and collective criteria can conflict. See the [1972 Prize summary](https://www.nobelprize.org/prizes/economic-sciences/1972/summary/).
- **Mechanism design.** Hurwicz, Maskin, and Myerson studied which institutions implement social objectives when information and incentives constrain behavior. See the [2007 Prize summary](https://www.nobelprize.org/prizes/economic-sciences/2007/summary/).
- **Matching and market design.** Gale and Shapley's deferred-acceptance idea became a foundation for stable matching; Roth and Shapley were recognized for stable allocations and market design. See [Gale and Shapley (1962)](https://doi.org/10.2307/2312726) and the [2012 Prize summary](https://www.nobelprize.org/prizes/economic-sciences/2012/summary/).
- **School-choice institutions.** The mechanism-design approach and Boston evidence are developed by [Abdulkadiroğlu and Sönmez (2003)](https://doi.org/10.1257/000282803322157061) and [Abdulkadiroğlu et al. (2005)](https://doi.org/10.1257/000282805774669637).

The sequence is conceptual rather than a claim that one field replaced another: game theory analyzes strategic interdependence; social choice articulates collective criteria; mechanism design asks which rules implement those criteria under information and incentive constraints.

## The minimum viable matching game

**Players.** Four students and three schools. If schools' priorities are policy inputs rather than strategic choices, the strategic players in this classroom model are the students. A richer model could make schools strategic too.

**Student strategy set.** Each student can submit multiple possible rank-order lists. There are therefore at least two players and at least two strategies per player; the displayed truthful list is one strategy, not the entire strategy set.

**Outcome and payoff.** A mechanism maps the submitted profile to an assignment. We use ordinal preferences: receiving a higher-ranked school is better. This is sufficient for the manipulation counterexample below but does not measure interpersonal welfare.

**Information.** Preferences and priorities are visible in this small teaching example. A research application must state which of them are known, private, estimated, or elicited.

| Student | 1st choice | 2nd choice | 3rd choice |
|---|---|---|---|
| Amina | Beacon | Aurora | Cedar |
| Bo | Beacon | Aurora | Cedar |
| Chen | Aurora | Beacon | Cedar |
| Dara | Aurora | Cedar | Beacon |

Every school uses priority **Amina ≻ Bo ≻ Chen ≻ Dara**. Capacities are Aurora 1, Beacon 1, and Cedar 2. This deliberately small example is a counterexample laboratory, not a model of any particular school district.

## Mechanism pseudocode · the one difference to remember

Let $U$ be students still seeking a seat, $P_s^r$ the round-$r$ proposals to school $s$, $H_s^r$ its held students, and $q_s$ its capacity.

| Phase | Boston / immediate acceptance | Gale–Shapley / student-proposing deferred acceptance |
|---|---|---|
| ⚙️ **Initialize** | $U\leftarrow N$; all seats open | $U\leftarrow N$; $H_s^0\leftarrow\varnothing$ for every school |
| → **Propose** | Each $i\in U$ applies to rank $r$ | Each rejected $i\in U$ proposes to the next untried school |
| 🏫 **School decision** | Permanently accept the highest-priority students in $P_s^r$ up to **remaining** capacity | From $H_s^{r-1}\cup P_s^r$, tentatively hold the top $q_s$ students and release the rest |
| ↺ **Continue** | Accepted students leave the process; rejected students try rank $r+1$ | Rejected or displaced students continue; held students wait and may still be displaced |
| 🏁 **Finalize** | Every acceptance is final immediately | Only when no proposal remains do all tentative holds become final |

**Critical difference:** Boston locks a seat now; deferred acceptance keeps the seat contestable until proposals stop.

### Baseline trace to predict before running

| Round | Proposals | Boston decision | Deferred-acceptance decision |
|---:|---|---|---|
| **1** | Amina → Beacon; Bo → Beacon; Chen → Aurora; Dara → Aurora | Beacon accepts Amina **finally**; Aurora accepts Chen **finally**. Amina and Chen leave. Bo and Dara continue. | Beacon holds Amina; Aurora holds Chen. Bo and Dara are rejected and continue. No hold is final. |
| **2** | Bo → Aurora; Dara → Cedar | Aurora is already full, so Bo is rejected. Cedar accepts Dara finally; Dara leaves. | Aurora compares held Chen with Bo, holds Bo, and displaces Chen. Cedar holds Dara. Chen continues. |
| **3** | Boston: Bo → Cedar. DA: Chen → Beacon. | Cedar accepts Bo finally; Boston stops. | Beacon keeps Amina and rejects Chen; Chen continues. |
| **4** | DA only: Chen → Cedar | — | Cedar holds Chen and Dara. With no proposal left, all holds become final. |

In [1]:
from copy import deepcopy
from html import escape
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
import pandas as pd

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

STUDENTS = ["Amina", "Bo", "Chen", "Dara"]
SCHOOLS = ["Aurora", "Beacon", "Cedar"]
CAPACITIES = {"Aurora": 1, "Beacon": 1, "Cedar": 2}
PREFERENCES = {"Amina": ["Beacon", "Aurora", "Cedar"], "Bo": ["Beacon", "Aurora", "Cedar"],
               "Chen": ["Aurora", "Beacon", "Cedar"], "Dara": ["Aurora", "Cedar", "Beacon"]}
PRIORITIES = {school: STUDENTS.copy() for school in SCHOOLS}

In [2]:
def priority_rank(priorities, school, student):
    """Smaller values mean higher school priority."""
    return priorities[school].index(student)

def new_assignment(students, schools):
    return {"by_student": dict.fromkeys(students), "by_school": {school: [] for school in schools}}

def snapshot(round_number, label, assignment, applications=None):
    return {"round": round_number, "label": label, "applications": deepcopy(applications or {}), "by_student": deepcopy(assignment["by_student"]), "by_school": deepcopy(assignment["by_school"])}

def run_boston(students, schools, capacities, preferences, priorities):
    """Boston: accept current-round proposals permanently; accepted students exit."""
    state, history = new_assignment(students, schools), []
    history.append(snapshot(0, "Start: every student is active", state))
    for rank in range(max(map(len, preferences.values()))):
        active = [student for student in students if state["by_student"][student] is None]
        applications = {school: [s for s in active if rank < len(preferences[s]) and preferences[s][rank] == school] for school in schools}
        for school in schools:
            vacancies = capacities[school] - len(state["by_school"][school])
            accepted = sorted(applications[school], key=lambda s: priority_rank(priorities, school, s))[:max(0, vacancies)]
            state["by_school"][school].extend(accepted)
            state["by_student"].update({student: school for student in accepted})
        history.append(snapshot(rank + 1, f"Round {rank + 1}: acceptances are final", state, applications))
    return state | {"history": history}

def run_deferred_acceptance(students, schools, capacities, preferences, priorities):
    """Gale–Shapley DA: reconsider old holds with new proposals; only the end is final."""
    state, next_choice = new_assignment(students, schools), dict.fromkeys(students, 0)
    history, round_number = [snapshot(0, "Start: no school holds a student", state)], 0
    while any(state["by_student"][s] is None and next_choice[s] < len(preferences[s]) for s in students):
        round_number += 1
        active = [s for s in students if state["by_student"][s] is None and next_choice[s] < len(preferences[s])]
        applications = {school: [] for school in schools}
        for student in active:
            school = preferences[student][next_choice[student]]
            next_choice[student] += 1
            applications[school].append(student)
        for school in schools:
            pool = state["by_school"][school] + applications[school]  # Old holds compete with new proposals.
            held = sorted(pool, key=lambda s: priority_rank(priorities, school, s))[:capacities[school]]
            state["by_student"].update({student: school if student in held else None for student in pool})
            state["by_school"][school] = held
        history.append(snapshot(round_number, f"Round {round_number}: holds are tentative", state, applications))
    history.append(snapshot(round_number + 1, "Final: tentative holds become assignments", state))
    return state | {"history": history}

def blocking_pairs(result, students, schools, capacities, preferences, priorities):
    """Return pairs that prefer one another to the displayed assignment."""
    pairs = []
    for student in students:
        current = result["by_student"][student]
        for school in schools:
            assigned = result["by_school"][school]
            student_prefers = school != current and (current is None or preferences[student].index(school) < preferences[student].index(current))
            school_accepts = len(assigned) < capacities[school] or any(priority_rank(priorities, school, student) < priority_rank(priorities, school, other) for other in assigned)
            if student_prefers and school_accepts:
                pairs.append((student, school))
    return pairs

def allocation_frame(result, label):
    return pd.DataFrame({"Student": list(result["by_student"]), label: [x or "Unmatched" for x in result["by_student"].values()]})

def round_story(result, index, mechanism):
    """Translate computed state changes into proposal → decision → continuation prose."""
    step = result["history"][index]
    if index == 0:
        return {"Proposals": "None yet", "School decisions": "No seats decided", "Who continues": "All students begin active"}
    previous, applications = result["history"][index - 1], step["applications"]
    if not applications:  # DA's finalization snapshot.
        final = "; ".join(f"{s} → {step['by_student'][s]}" for s in STUDENTS)
        return {"Proposals": "No proposal remains", "School decisions": f"All holds become final: {final}", "Who continues": "The mechanism stops"}
    proposals = "; ".join(f"{student} → {school}" for school in SCHOOLS for student in applications.get(school, [])) or "No new proposal"
    decisions = []
    for school in SCHOOLS:
        pool = list(dict.fromkeys(previous["by_school"][school] + applications.get(school, [])))
        selected = step["by_school"][school]
        accepted = selected if mechanism != "Boston" else [s for s in selected if s not in previous["by_school"][school]]
        rejected = [student for student in pool if student not in selected]
        if applications.get(school) or (mechanism == "DA" and rejected):
            verb = "FINAL accept" if mechanism == "Boston" else "tentatively hold"
            decisions.append(f"{school}: {verb} {', '.join(accepted) or 'no one'}; reject/release {', '.join(rejected) or 'no one'}")
    assigned, active = ([s for s in STUDENTS if step["by_student"][s] is not None], [s for s in STUDENTS if step["by_student"][s] is None])
    continuation = (f"Permanently assigned and out: {', '.join(assigned) or 'none'}. Next round: {', '.join(active) or 'none'}." if mechanism == "Boston" else f"Held, but not final: {', '.join(assigned) or 'none'}. Rejected/displaced and proposing next: {', '.join(active) or 'none'}.")
    return {"Proposals": proposals, "School decisions": " | ".join(decisions), "Who continues": continuation}

def history_frame(result, mechanism):
    return pd.DataFrame([{"Round": step["round"], **round_story(result, i, mechanism)} for i, step in enumerate(result["history"])])

In [3]:
boston = run_boston(STUDENTS, SCHOOLS, CAPACITIES, PREFERENCES, PRIORITIES)
da = run_deferred_acceptance(STUDENTS, SCHOOLS, CAPACITIES, PREFERENCES, PRIORITIES)

comparison = allocation_frame(boston, "Boston").merge(allocation_frame(da, "Deferred acceptance"), on="Student")
display(comparison)
display(HTML("<h3>Boston: permanent decisions round by round</h3>")); display(history_frame(boston, "Boston"))
display(HTML("<h3>Deferred acceptance: holds, releases, and finalization</h3>")); display(history_frame(da, "DA"))
print("Boston blocking pairs:", blocking_pairs(boston, STUDENTS, SCHOOLS, CAPACITIES, PREFERENCES, PRIORITIES))
print("DA blocking pairs:", blocking_pairs(da, STUDENTS, SCHOOLS, CAPACITIES, PREFERENCES, PRIORITIES))

assert boston["by_student"]["Bo"] == "Cedar" and da["by_student"]["Bo"] == "Aurora"
assert blocking_pairs(boston, STUDENTS, SCHOOLS, CAPACITIES, PREFERENCES, PRIORITIES) == [("Bo", "Aurora")]
assert blocking_pairs(da, STUDENTS, SCHOOLS, CAPACITIES, PREFERENCES, PRIORITIES) == []

,Student,Boston,Deferred acceptance
0,Amina,Beacon,Beacon
1,Bo,Cedar,Aurora
2,Chen,Aurora,Cedar
3,Dara,Cedar,Cedar


,Round,Proposals,School decisions,Who continues
0,0,None yet,No seats decided,All students begin active
1,1,Chen → Aurora; Dara → Aurora; Amina → Beacon; ...,Aurora: FINAL accept Chen; reject/release Dara...,"Permanently assigned and out: Amina, Chen. Nex..."
2,2,Bo → Aurora; Dara → Cedar,Aurora: FINAL accept no one; reject/release Bo...,"Permanently assigned and out: Amina, Chen, Dar..."
3,3,Bo → Cedar,Cedar: FINAL accept Bo; reject/release no one,"Permanently assigned and out: Amina, Bo, Chen,..."


,Round,Proposals,School decisions,Who continues
0,0,None yet,No seats decided,All students begin active
1,1,Chen → Aurora; Dara → Aurora; Amina → Beacon; ...,Aurora: tentatively hold Chen; reject/release ...,"Held, but not final: Amina, Chen. Rejected/dis..."
2,2,Bo → Aurora; Dara → Cedar,Aurora: tentatively hold Bo; reject/release Ch...,"Held, but not final: Amina, Bo, Dara. Rejected..."
3,3,Chen → Beacon,Beacon: tentatively hold Amina; reject/release...,"Held, but not final: Amina, Bo, Dara. Rejected..."
4,4,Chen → Cedar,"Cedar: tentatively hold Chen, Dara; reject/rel...","Held, but not final: Amina, Bo, Chen, Dara. Re..."
5,5,No proposal remains,All holds become final: Amina → Beacon; Bo → A...,The mechanism stops


Boston blocking pairs: [('Bo', 'Aurora')]
DA blocking pairs: []


### Interpret the result in three voices

- **Game theorist:** Under Boston, Bo's truthful report leads to Cedar. Is there another report that improves Bo's assignment while others' reports stay fixed?
- **Social-choice researcher:** The Boston allocation violates stability in this instance because Bo prefers Aurora to Cedar and Aurora gives higher priority to Bo than to Chen. Which social criteria should be decisive, and why?
- **Mechanism designer:** Student-proposing deferred acceptance changes provisional-hold rules so the later proposal from Bo can displace Chen at Aurora. The observed instance is stable. The general strategy-proofness and stability claims require the theorem's standard assumptions; the simulation illustrates rather than proves them.

In [4]:
def animate_history(result, title, mechanism):
    """Return an accessible CSS animation of proposal → decision → continuation."""
    slug, frames = f"matching-{mechanism.lower()}", []
    for index, step in enumerate(result["history"]):
        story = round_story(result, index, mechanism)
        cards = "".join(f"<article><b>{escape(label)}</b><span>{escape(text)}</span></article>" for label, text in story.items())
        assignments = "".join(f"<li><b>{escape(student)}</b><span>→ {escape(school or 'active')}</span></li>" for student, school in step["by_student"].items())
        frames.append(f"""<section class="matching-frame"><p class="round">{escape(step['label'])}</p><div class="phase-cards">{cards}</div><ul>{assignments}</ul></section>""")
    interval, count = 2.4, len(frames)
    delays = "".join(f"#{slug} .matching-frame:nth-child({i+1}){{animation-delay:-{i*interval:.1f}s}}" for i in range(count))
    return HTML(f"""<style>
      #{slug}{{background:#071326;color:#edf7ff;border:1px solid #214364;border-radius:18px;padding:20px;font-family:Arial,sans-serif;overflow:hidden}}
      #{slug} .stage{{display:grid;min-height:330px}} #{slug} .matching-frame{{grid-area:1/1;opacity:0;transform:translateX(18px);animation:{slug}-cycle {interval*count:.1f}s infinite}}
      #{slug} .round{{color:#ffcc66;font-size:18px;font-weight:700}} #{slug} .phase-cards{{display:grid;grid-template-columns:repeat(3,1fr);gap:8px}}
      #{slug} article,#{slug} li{{background:#10263f;border:1px solid #214364;border-radius:10px;padding:10px}} #{slug} article b,#{slug} article span{{display:block}}
      #{slug} article b{{color:#35d4ff;font-size:11px;text-transform:uppercase}} #{slug} article span{{margin-top:6px;font-size:13px;line-height:1.4}}
      #{slug} ul{{display:grid;grid-template-columns:repeat(4,1fr);gap:7px;padding:0;list-style:none}} #{slug} li{{display:flex;justify-content:space-between;font-size:12px}}
      {delays}@keyframes {slug}-cycle{{0%,{max(8,100/count-3):.1f}%{{opacity:1;transform:none}}{max(10,100/count-1):.1f}%,100%{{opacity:0;transform:translateX(-18px)}}}}
      @media(max-width:700px){{#{slug} .phase-cards,#{slug} ul{{grid-template-columns:1fr}}}}
      @media(prefers-reduced-motion:reduce){{#{slug} .matching-frame{{display:none;animation:none}}#{slug} .matching-frame:first-child{{display:block;opacity:1;transform:none}}}}
    </style><div id="{slug}" class="matching-animation" role="img" aria-label="{escape(title)}: animated matching rounds">
      <p><b>{escape(title)}</b> · proposals → school decisions → who continues</p><div class="stage">{''.join(frames)}</div></div>""")

display(animate_history(boston, "Boston mechanism: final acceptance each round", "Boston"))
display(animate_history(da, "Gale–Shapley: tentative holds until the end", "DA"))

In [5]:
# First, preserve the Boston manipulation counterexample.
strategic_preferences = deepcopy(PREFERENCES)
strategic_preferences["Bo"] = ["Aurora", "Beacon", "Cedar"]
truthful_boston = run_boston(STUDENTS, SCHOOLS, CAPACITIES, PREFERENCES, PRIORITIES)
strategic_boston = run_boston(STUDENTS, SCHOOLS, CAPACITIES, strategic_preferences, PRIORITIES)
true_rank = {school: rank for rank, school in enumerate(PREFERENCES["Bo"])}
assert true_rank[strategic_boston["by_student"]["Bo"]] < true_rank[truthful_boston["by_student"]["Bo"]]

# Then let students change exactly one report and recompute either mechanism.
experiment_student = widgets.Dropdown(options=STUDENTS, value="Bo", description="Student")
experiment_first_choice = widgets.Dropdown(options=SCHOOLS, value="Aurora", description="Move to #1")
experiment_mechanism = widgets.ToggleButtons(options=[("Boston", "Boston"), ("Gale–Shapley DA", "DA")], description="Rule")
experiment_button, experiment_output = widgets.Button(description="Run changed report", button_style="primary"), widgets.Output()

def run_experiment(_=None):
    preferences = deepcopy(PREFERENCES)
    student, first = experiment_student.value, experiment_first_choice.value
    preferences[student] = [first] + [school for school in preferences[student] if school != first]
    runner = run_boston if experiment_mechanism.value == "Boston" else run_deferred_acceptance
    result = runner(STUDENTS, SCHOOLS, CAPACITIES, preferences, PRIORITIES)
    with experiment_output:
        clear_output(wait=True)
        display(allocation_frame(result, experiment_mechanism.value))
        display(history_frame(result, experiment_mechanism.value))
        print("Blocking pairs:", blocking_pairs(result, STUDENTS, SCHOOLS, CAPACITIES, preferences, PRIORITIES))
    return result

experiment_button.on_click(run_experiment)
display(widgets.VBox([widgets.HTML("<b>Change one submitted ranking, predict, then run.</b>"), experiment_student,
                      experiment_first_choice, experiment_mechanism, experiment_button, experiment_output]))

Interactive controls: run this cell in Colab. Saved demonstrations are shown above.

## Your change-one-input investigation

School choice uses **ordinal rankings**, not cardinal payoff numbers. Change one submitted ranking in the control panel above; use Notebooks 01–02 when the research question requires editable numerical payoffs.

1. **Prediction:** which proposal, school decision, and final assignment will change?
2. **Game-theory interpretation:** which student's strategy or incentive changed?
3. **Social-choice interpretation:** which collective criterion improved or worsened?
4. **Mechanism-design interpretation:** did the blocking pair or manipulation counterexample survive?
5. **Verification:** compare the computed round trace with your prediction.
6. **Boundary:** what does this single instance fail to establish?

### Human-led AI protocol

Record your reasoning first. Then ask an AI assistant for one counterexample or edge case. Verify every mathematical and bibliographic claim against the code, a cited paper, or an authoritative source; record what you accepted, rejected, and revised.

## Transfer to the final project

No matter your research topic, your future-research roadmap should include:

- **Joint perspective:** show how game theory, social choice, and mechanism design ask different but connected questions about the same problem.
- **Application transfer:** discuss what your model reveals when interpreted as an auction/allocation problem and as a voting/matching/collective-decision problem. State where the analogy breaks.
- **Computational artifact:** document pseudocode, executable code, simulation results, assumptions, edge cases, and reproducibility instructions in GitHub.
- **Economic and behavioral account:** explain incentives and welfare, then identify where actual behavior may depart from the formal model and what evidence would distinguish explanations.
- **Interdisciplinary frontier:** identify an intellectual contribution and practical impact across computer science, economics, and behavioral science; state open questions and who could benefit or bear risk.

Your notebook output is evidence only when a reader can reproduce it, inspect the mechanism, and understand what would falsify your claim.